<div class="title-slide">

# Make the Most of What You’ve Got: Deep Learning Optimization with PyTorch Profiler and Holistic Trace Analysis 

## Collin Wilson

</div>

# Getting started with `sn-torch-lab`

### This talk is given in a Jupyter notebook format. You can download it and all the materials to recreate the demo on GitHub:

```
git clone https://github.com/cwilson51/sn-torch-lab.git
```

### You can then install the repository as a package in a virtual environment. On Alliance Clusters:

```bash
module load StdEnv/2023 python/3.11 scipy-stack/2026a
virtualenv --no-download torch-labENV && source torch-labENV/bin/activate
pip install --no-index upgrade pip

git clone https://github.com/cwilson51/sn-torch-lab.git
pip install --no-index ./sn-torch-lab
```

<div class="pt-slide">
    
# What we'll cover today

- ## PyTorch Profiler revisited
    - ### Inline profiling
    - ### Generating traces
- ## Examining traces and simple optimizations
- ## Holistic trace analysis (HTA)
    - ### Single GPU
    - ### Distributed workloads
- ## Trace Analysis with Brine
</div>

<div class="pt-slide">
    
# PyTorch Profiler

- ## Built-in module of PyTorch
- ## Can profile run times, memory and generate detailed traces of PyTorch code
    - ### see layer-by-layer breakdown of model performance
- ## Both CPU and GPU profiling
- ## Code modification required - wrap code with a context manager
- ## Works for NVIDIA and AMD GPUs; separate tool for Apple MPS
- ## Today’s live profiles are with the H100 1g.10gb MIG flavour on Nibi
</div>

# Working example - CPU

## ResNet50 on random inputs

In [10]:
import torch
from torch.utils.data import DataLoader
from torch.profiler import profile, record_function, ProfilerActivity, schedule

from torchvision.models import resnet50
from torchvision.datasets import FakeData
from torchvision import transforms

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    device = "cuda"
    activities += [ProfilerActivity.CUDA]

### Model and optimizer

In [5]:
model = resnet50()
model.to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

### Data

In [11]:
train_ds = FakeData(
    size=10_000,                 # number of samples
    image_size=(3, 224, 224),
    num_classes=1000,
    transform=transforms.ToTensor(),
)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                          num_workers=4, pin_memory=True)

### Run a simple profile

In [12]:
inputs, labels = next(iter(train_loader))

with profile(activities=activities, record_shapes=True) as prof:

    model.train()
    with record_function("model_training"):
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
    model.eval()
    with record_function("model_inference"):
        model(inputs.cuda())

/Users/collinwilson/projects/sn-torch-lab/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
USDT:2026-11-17 14:21:21 87887:18713097 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-11-17 14:21:29 87887:18713097 SyncActivityProfilerHandler.cpp:46] profiler_stop


KeyboardInterrupt: 

### Results:

In [ ]:
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))